# ST-Reason · 08 Evaluation metrics

Reads everything the earlier notebooks saved on Drive and computes the paper's metrics. **CPU only** (no GPU needed), a few minutes. It changes nothing it reads, so it can be re-run at any time; each run overwrites its own result files.

| Section | Metrics | Output in `outputs/metrics/` |
| --- | --- | --- |
| 2. QA accuracy | accuracy per run with 95% cluster-bootstrap CIs (resampling videos), valid-reply rate, **lenient** accuracy (a letter recovered from an unreadable reply) | `qa_runs.csv` |
| 3. By question type | NExT-QA causal / temporal / descriptive; STAR Sequence / Interaction | `qa_by_type.csv` |
| 4. Cost | LLM calls, prompt and generated tokens, seconds per question; memory size per format | `qa_runs.csv`, `memory_size.csv` |
| 5. Verification (strategy D) | how often the verifier and the ID check pass, how often a second answer was needed, accuracy when supported vs not | `verification.csv` |
| 6. Grounding (NExT-GQA) | do the memory items an answer cites lie inside the human-marked moment? IoP, Acc@GQA | `grounding.csv` |
| 7. CV stage | STAR events vs Charades ground truth (temporal precision / recall / F1); detection AP on VidOR where ground truth exists; track statistics | `cv_metrics.json` |

All answers in one tidy table go to `answers_all.csv`, which notebook 09 uses for the statistical tests.

**Which questions:** the shards in `EVAL_SHARDS` (0 and 1, as run in 05–07). Accuracy here is per run on the questions that run answered; paired comparisons on exactly the same questions are done in 09.

In [3]:
# --- Parameters ----------------------------------------------------------------------------------
EVAL_SHARDS = [0, 1]
N_SHARDS = 4
N_BOOT = 2000              # bootstrap resamples for confidence intervals
TIOU = 0.3                 # temporal IoU for matching an event to a ground-truth action

In [4]:
# --- 0. Bootstrap --------------------------------------------------------------------------
import os, glob
ST_NOTEBOOK = '08_eval_metrics'
ST_QUIET = True
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB
import pandas as pd, numpy as np
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 200)
MANIFEST_PATH = P['runs'] / 'manifest_08.csv'
DRIVE_LOG_PATH = LOG_PATH
LOG_PATH = P['local'] / 'logs' / LOG_PATH.name
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT = P['metrics'] if P['metrics'].exists() else P['pred']       # files only, no new folders on the shared Drive
WORK = P['local'] / 'eval'
WORK.mkdir(parents=True, exist_ok=True)
RUN_LOG = Run(params={'step': 'eval_metrics', 'shards': EVAL_SHARDS, 'n_boot': N_BOOT})
def shard_of(uid):
    return int(hashlib.sha1(str(uid).encode()).hexdigest(), 16) % N_SHARDS
def save_csv(df, name):
    tmp = WORK / name
    df.to_csv(tmp, index=False)
    shutil.copyfile(tmp, OUT / name)
print('results go to', OUT)

Mounted at /content/drive
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local
config hash: 2a9e0e4f87
[05:46:42] INFO  run started: 08_eval_metrics-20261019-054634-ea4ecb
results go to /content/drive/MyDrive/ST-Reason/outputs/metrics


## 1. All answers

Every answer zip from notebooks 05, 06 and 07 in `outputs/predictions/` is read. If the same question appears twice in one run (a redo), the newest copy is kept. Question details (type, video, grounding segment) come from the frozen sets.

In [5]:
def zip_time(p):
    m = re.search(r'_(\d{8}-\d{6})_', p.name)
    return m.group(1) if m else ''
ZIPS = sorted(P['pred'].glob('0[567]_*.zip'), key=zip_time, reverse=True)     # newest first
rows, seen, bad = [], set(), []
for z in ZIPS:
    try:
        with zipfile.ZipFile(z) as zf:
            for m in zf.namelist():
                if not m.endswith('.jsonl'):
                    continue
                for line in zf.read(m).decode().splitlines():
                    if not line.strip():
                        continue
                    r = json.loads(line)
                    k = (r['run'], r['uid'])
                    if k in seen:
                        continue
                    seen.add(k); r['_zip'] = z.name[:2]; rows.append(r)
    except (zipfile.BadZipFile, OSError) as e:
        bad.append(z.name)
A = pd.DataFrame(rows)
print(f'{len(ZIPS)} answer zips, {len(A)} answers, {A.run.nunique()} runs' + (f'; damaged zips skipped: {bad}' if bad else ''))

def read_set(name):
    f = P['subsets'] / f'{name}.csv'
    return pd.read_csv(f, dtype={'video_id': str}) if f.exists() else pd.DataFrame()
NQ = pd.concat([read_set('main_grid_eval'), read_set('dev')]).drop_duplicates('uid').set_index('uid')
SQ = pd.concat([read_set('star_main'), read_set('star_gap')]).drop_duplicates('uid').set_index('uid')

A['base_uid'] = A.uid.str.split('__').str[0]
A['version'] = np.where(A.uid.str.contains('__'), A.uid.str.split('__').str[-1], 'orig')
A['shard'] = A.base_uid.map(shard_of)
A['dataset'] = np.where(A.base_uid.str.startswith('star_'), 'star', 'nextqa')
A['video'] = A.base_uid.map(NQ.video_id.to_dict()).fillna(A.base_uid.map(SQ.video_id.to_dict()) if len(SQ) else np.nan)
A['group'] = A.base_uid.map(NQ.group.to_dict()) if 'group' in NQ else np.nan
A.loc[A.dataset == 'star', 'group'] = A.loc[A.dataset == 'star', 'base_uid'].map(SQ.type.to_dict()) if len(SQ) else np.nan
for c in ['calls', 'prompt_tokens', 'gen_tokens', 'seconds', 'mem_tokens', 'retrieved_tokens', 'raw', 'evidence', 'model']:
    if c not in A:
        A[c] = np.nan
outside = A[~A.shard.isin(EVAL_SHARDS)]
if len(outside):
    print(f'note: {len(outside)} answers belong to other shards and are left out: {outside.run.value_counts().head().to_dict()}')
A = A[A.shard.isin(EVAL_SHARDS)].reset_index(drop=True)
print(A.groupby('run').size().rename('answers').to_string())

60 answer zips, 4188 answers, 33 runs
run
dev.M4.A               78
dev.M4.A-SC            78
dev.M4.D               78
gap_pause10.M1.D       32
gap_pause10.M4.D       61
gap_pause25.M1.D       32
gap_pause25.M4.D       32
gap_sham10.M1.D        32
gap_sham10.M4.D        61
gap_sham25.M1.D        32
gap_sham25.M4.D        32
main.BLIND.A          193
main.CAP.A            100
main.M1.A             193
main.M1.D             193
main.M4.A             193
main.M4.A.q4b         193
main.M4.A.q4b-lora    193
main.M4.D             193
main.VA.A             100
main.VLM.F16          193
star.BLIND.A          190
star.CAP.A             94
star.M1.A             190
star.M1.D             190
star.M4.A             190
star.M4.A.q4b         190
star.M4.A.q4b-lora    190
star.M4.D             190
star.ORACLE_M1.D       94
star.ORACLE_M4.D       94
star.VA.A              94
star.VLM.F16          190


**Lenient scoring.** An answer counts as invalid when its reply could not be read as the requested JSON (strict scoring counts it wrong). Lenient scoring recovers a letter from the saved reply text (the first `"answer": "X"`, else the first standalone option letter) and scores that. Strict accuracy is the main result; lenient accuracy shows how much of a difference between runs is only the reply format (e.g. M1's low valid-reply rate).

In [6]:
LET = 'ABCDE'
def lenient_letter(raw, n_opt):
    if not isinstance(raw, str) or not raw:
        return None
    ok = LET[:n_opt]
    m = re.search(r'"?answer"?\s*[:=]\s*"?\(?([A-E])\b', raw, flags=re.I)
    if m and m.group(1).upper() in ok:
        return m.group(1).upper()
    m = re.search(r'\b([A-E])\b', raw)
    return m.group(1) if m and m.group(1) in ok else None
n_opt = A.base_uid.map(SQ.n_options.to_dict() if 'n_options' in SQ else {}).fillna(5).astype(int)
A['pred_lenient'] = [p if isinstance(p, str) and p else lenient_letter(r, n) for p, r, n in zip(A.pred, A.get('raw'), n_opt)]
A['correct'] = A.correct.astype(bool)
A['correct_lenient'] = A.pred_lenient == A.gold
A['valid_json'] = A.valid_json.astype(bool)
print('replies recovered by lenient scoring:', int((~A.valid_json & A.pred_lenient.notna()).sum()), 'of', int((~A.valid_json).sum()), 'invalid')

replies recovered by lenient scoring: 239 of 248 invalid


## 2. QA accuracy per run

**Confidence intervals** are 95% percentile intervals from a cluster bootstrap: videos (not questions) are resampled, because questions about the same video are not independent.

In [7]:
RNG = np.random.default_rng(CFG['seed'])
def cluster_ci(df, col):
    g = df.groupby('video')[col].agg(['sum', 'count'])
    s, n = g['sum'].to_numpy(float), g['count'].to_numpy(float)
    if len(g) < 2:
        return np.nan, np.nan
    idx = RNG.integers(0, len(g), size=(N_BOOT, len(g)))
    acc = s[idx].sum(1) / n[idx].sum(1)
    return np.percentile(acc, 2.5), np.percentile(acc, 97.5)

def parse_run(rid):
    p = rid.split('.')
    return {'qset': p[0], 'format': p[1] if len(p) > 1 else '', 'strategy': p[2] if len(p) > 2 else '',
            'variant': '.'.join(p[3:]) if len(p) > 3 else ''}

R = []
for rid, d in A.groupby('run'):
    lo, hi = cluster_ci(d, 'correct')
    llo, lhi = cluster_ci(d, 'correct_lenient')
    R.append({'run': rid, **parse_run(rid), 'model': d.model.mode().iat[0] if d.model.notna().any() else '',
              'n': len(d), 'videos': d.video.nunique(), 'accuracy': d.correct.mean(), 'ci_low': lo, 'ci_high': hi,
              'valid': d.valid_json.mean(), 'accuracy_lenient': d.correct_lenient.mean(), 'len_ci_low': llo, 'len_ci_high': lhi,
              'calls_per_q': d.calls.mean(), 'prompt_tok_per_q': d.prompt_tokens.mean(), 'gen_tok_per_q': d.gen_tokens.mean(),
              's_per_q': d.seconds.mean(), 'mem_tokens': d.mem_tokens.mean()})
QR = pd.DataFrame(R).sort_values(['qset', 'format', 'strategy', 'variant']).reset_index(drop=True)
save_csv(QR.round(4), 'qa_runs.csv')
show = QR[['run', 'model', 'n', 'accuracy', 'ci_low', 'ci_high', 'valid', 'accuracy_lenient', 'calls_per_q', 's_per_q']].copy()
for c in ['accuracy', 'ci_low', 'ci_high', 'valid', 'accuracy_lenient']:
    show[c] = show[c].map(lambda x: f'{x:.3f}' if x == x else '')
show['calls_per_q'] = show.calls_per_q.round(2); show['s_per_q'] = show.s_per_q.round(1)
print(show.to_string(index=False))

               run                     model   n accuracy ci_low ci_high valid accuracy_lenient  calls_per_q  s_per_q
          dev.M4.A             Qwen/Qwen3-8B  78    0.500  0.392   0.611 0.897            0.526         1.13     12.0
       dev.M4.A-SC             Qwen/Qwen3-8B  78    0.487  0.376   0.597 0.910            0.538         3.00     27.0
          dev.M4.D             Qwen/Qwen3-8B  78    0.513  0.385   0.633 0.962            0.526         4.29     20.9
  gap_pause10.M1.D             Qwen/Qwen3-8B  32    0.438  0.250   0.625 0.844            0.469         5.19     22.7
  gap_pause10.M4.D             Qwen/Qwen3-8B  61    0.541  0.410   0.678 0.967            0.541         4.15     18.3
  gap_pause25.M1.D             Qwen/Qwen3-8B  32    0.438  0.281   0.594 0.812            0.531         5.06     23.3
  gap_pause25.M4.D             Qwen/Qwen3-8B  32    0.500  0.344   0.688 0.969            0.500         4.16     18.6
   gap_sham10.M1.D             Qwen/Qwen3-8B  32    0.40

## 3. By question type

In [8]:
T = (A.groupby(['run', 'group']).agg(n=('correct', 'size'), accuracy=('correct', 'mean'),
                                     accuracy_lenient=('correct_lenient', 'mean')).reset_index())
save_csv(T.round(4), 'qa_by_type.csv')
piv = T.pivot(index='run', columns='group', values='accuracy').round(3)
cnt = T.pivot(index='run', columns='group', values='n')
print(piv.to_string())
print('\nquestions per type:\n', cnt.max().astype('Int64').to_string())

group               Interaction  Sequence  causal  temporal
run                                                        
dev.M4.A                    NaN       NaN   0.439     0.568
dev.M4.A-SC                 NaN       NaN   0.415     0.568
dev.M4.D                    NaN       NaN   0.488     0.541
gap_pause10.M1.D            NaN     0.438     NaN       NaN
gap_pause10.M4.D            NaN     0.541     NaN       NaN
gap_pause25.M1.D            NaN     0.438     NaN       NaN
gap_pause25.M4.D            NaN     0.500     NaN       NaN
gap_sham10.M1.D             NaN     0.406     NaN       NaN
gap_sham10.M4.D             NaN     0.525     NaN       NaN
gap_sham25.M1.D             NaN     0.438     NaN       NaN
gap_sham25.M4.D             NaN     0.469     NaN       NaN
main.BLIND.A                NaN       NaN   0.396     0.446
main.CAP.A                  NaN       NaN   0.423     0.583
main.M1.A                   NaN       NaN   0.416     0.424
main.M1.D                   NaN       Na

## 4. Cost and memory size

Calls, tokens and seconds per question are in `qa_runs.csv` (section 2). Memory size per format comes from notebook 04's `memory_stats.csv`.

In [9]:
ms = P['memory'] / 'memory_stats.csv'
if ms.exists():
    M = pd.read_csv(ms)
    MS = (M.groupby(['group', 'format']).agg(memories=('key', 'size'), tokens_median=('tokens', 'median'),
                                            tokens_mean=('tokens', 'mean'), items_median=('n_items', 'median'))
          .reset_index())
    save_csv(MS.round(1), 'memory_size.csv')
    print(MS.pivot(index='format', columns='group', values='tokens_median').round(0).to_string())
else:
    print('memory_stats.csv not found in outputs/memory/: skipped')
cost = QR[['run', 'calls_per_q', 'prompt_tok_per_q', 'gen_tok_per_q', 's_per_q']].round(1)
print('\n', cost.to_string(index=False))

group         dev  intentqa    main   star
format                                    
CAP         286.0     162.0   241.0  106.0
M1         1467.0    1183.0  1137.0  518.0
M2          836.0     597.0   691.0  307.0
M3         1611.0    1202.0  1230.0  598.0
M4         2431.0    1967.0  2006.0  968.0
ORACLE_M1     NaN       NaN     NaN  402.0
ORACLE_M2     NaN       NaN     NaN  156.0
ORACLE_M3     NaN       NaN     NaN  431.0
ORACLE_M4     NaN       NaN     NaN  690.0
VA          673.0     507.0   543.0  266.0

                run  calls_per_q  prompt_tok_per_q  gen_tok_per_q  s_per_q
          dev.M4.A          1.1            2997.3           39.5     12.0
       dev.M4.A-SC          3.0            7759.3           93.2     27.0
          dev.M4.D          4.3            3252.3          324.1     20.9
  gap_pause10.M1.D          5.2            3131.6          447.8     22.7
  gap_pause10.M4.D          4.1            2796.1          301.1     18.3
  gap_pause25.M1.D          5.1       

## 5. Verification (strategy D)

For every D answer: did the verifier say the cited items support it, did every cited ID exist among the shown items (symbolic check), and was the question answered a second time with a larger retrieval budget. Accuracy is split by the verifier's verdict, to show whether "supported" is a useful confidence signal.

In [10]:
V = []
for rid, d in A[A.strategy.isin(['D', 'D-X'])].groupby('run'):
    sup = d.verifier_supported.fillna(False).astype(bool) if 'verifier_supported' in d else pd.Series(False, index=d.index)
    sym = d.symbolic_ok.fillna(False).astype(bool) if 'symbolic_ok' in d else pd.Series(False, index=d.index)
    re_ = d.reanswered.fillna(False).astype(bool) if 'reanswered' in d else pd.Series(False, index=d.index)
    V.append({'run': rid, 'n': len(d), 'verifier_supported': sup.mean(), 'ids_valid': sym.mean(), 'reanswered': re_.mean(),
              'acc_if_supported': d.correct[sup].mean() if sup.any() else np.nan,
              'acc_if_not_supported': d.correct[~sup].mean() if (~sup).any() else np.nan,
              'evidence_items': d.evidence.map(lambda e: len(e) if isinstance(e, list) else 0).mean(),
              'retrieved_tokens': d.retrieved_tokens.mean()})
VR = pd.DataFrame(V)
if len(VR):
    save_csv(VR.round(4), 'verification.csv')
    print(VR.round(3).to_string(index=False))

/tmp/ipykernel_3820/616410466.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sup = d.verifier_supported.fillna(False).astype(bool) if 'verifier_supported' in d else pd.Series(False, index=d.index)
/tmp/ipykernel_3820/616410466.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sym = d.symbolic_ok.fillna(False).astype(bool) if 'symbolic_ok' in d else pd.Series(False, index=d.index)
/tmp/ipykernel_3820/616410466.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call resu

             run   n  verifier_supported  ids_valid  reanswered  acc_if_supported  acc_if_not_supported  evidence_items  retrieved_tokens
        dev.M4.D  78               0.205      0.846       0.846             0.625                 0.484           3.397           600.051
gap_pause10.M1.D  32               0.250      0.000       1.000             0.500                 0.417           2.000           472.438
gap_pause10.M4.D  61               0.197      0.885       0.803             0.500                 0.551           2.656           571.607
gap_pause25.M1.D  32               0.250      0.000       1.000             0.375                 0.458           1.844           472.281
gap_pause25.M4.D  32               0.188      0.844       0.844             0.500                 0.500           3.469           571.719
 gap_sham10.M1.D  32               0.281      0.000       1.000             0.333                 0.435           2.188           466.500
 gap_sham10.M4.D  61              

## 6. Grounding on NExT-GQA

NExT-GQA marks, for most main-grid questions, the moment of the video that answers it. For each answer we take the memory items it cites as evidence, and the time span they cover (an item with a single time gets ±0.5 s). **IoP** (intersection over prediction) is the share of that span inside the marked moment; **Acc@GQA** counts an answer as right only if it is correct *and* IoP ≥ 0.5, as in the NExT-GQA paper. For strategy D the same is reported for the retrieved items (did retrieval reach the right moment at all).

Item times come from the memory files of notebook 04 (only the needed ones are unpacked).

In [11]:
mem_zips = sorted(P['memory'].glob('04_memories_*.zip'))
G_rows = []
GQ = NQ[NQ.get('has_ground', pd.Series(False, index=NQ.index)).fillna(False).astype(bool)] if len(NQ) else NQ
if mem_zips and len(GQ):
    segs = {u: json.loads(s) for u, s in GQ.segments.items() if isinstance(s, str) and s.strip()}
    G = A[(A.dataset == 'nextqa') & A.base_uid.isin(segs) & A.format.str.match(r'^M[1-4]$')]
    need = {(f, k) for f, k in zip(G.format, G.key)}
    items = {}
    with zipfile.ZipFile(mem_zips[-1]) as zf:
        names = set(zf.namelist())
        for f, k in need:
            m = f'{f}/{k}.json'
            if m in names:
                items[(f, k)] = {it['id']: (float(it['t0']), float(it['t1'])) for it in json.loads(zf.read(m))['items']}
    def span(ids, f, k):
        tt = [items.get((f, k), {}).get(str(i)) for i in (ids if isinstance(ids, list) else [])]
        tt = [t for t in tt if t]
        if not tt:
            return None
        a, b = min(t[0] for t in tt), max(t[1] for t in tt)
        return (a - 0.5, b + 0.5) if b - a < 1e-6 else (a, b)
    def iop(sp, gt):
        if sp is None:
            return np.nan
        inter = sum(max(0.0, min(sp[1], e) - max(sp[0], s)) for s, e in gt)
        return min(1.0, inter / max(1e-6, sp[1] - sp[0]))
    G = G.assign(iop=[iop(span(e, f, k), segs[u]) for e, f, k, u in zip(G.evidence, G.format, G.key, G.base_uid)])
    if 'retrieved' in G:
        G = G.assign(ret_hit=[iop(span(r, f, k), segs[u]) > 0 if isinstance(r, list) and r else np.nan
                              for r, f, k, u in zip(G.retrieved, G.format, G.key, G.base_uid)])
    for rid, d in G.groupby('run'):
        cited = d.iop.notna()
        G_rows.append({'run': rid, 'n_grounded_q': len(d), 'with_citation': cited.mean(), 'mean_iop': d.iop.mean(),
                       'iop>=0.5': (d.iop >= 0.5).mean(), 'acc': d.correct.mean(),
                       'acc@gqa': (d.correct & (d.iop >= 0.5)).mean(),
                       'retrieval_hits_moment': d.ret_hit.mean() if 'ret_hit' in d else np.nan})
GR = pd.DataFrame(G_rows)
if len(GR):
    save_csv(GR.round(4), 'grounding.csv')
    print(GR.round(3).to_string(index=False))
else:
    print('no grounded questions or no memory zip found: grounding skipped')

               run  n_grounded_q  with_citation  mean_iop  iop>=0.5   acc  acc@gqa  retrieval_hits_moment
          dev.M4.A            78          0.897     0.387     0.346 0.500    0.192                    NaN
       dev.M4.A-SC            78          0.872     0.395     0.359 0.487    0.205                    NaN
          dev.M4.D            78          0.962     0.375     0.359 0.513    0.192                  1.000
         main.M1.A           193          0.000       NaN     0.000 0.420    0.000                    NaN
         main.M1.D           193          0.000       NaN     0.000 0.332    0.000                  0.979
         main.M4.A           193          0.948     0.420     0.389 0.544    0.228                    NaN
     main.M4.A.q4b           193          0.876     0.360     0.259 0.456    0.155                    NaN
main.M4.A.q4b-lora           193          0.000       NaN     0.000 0.549    0.000                    NaN
         main.M4.D           193          0.97

## 7. CV stage

**Events vs ground truth (STAR).** For every original STAR clip, the cleaned events of notebook 03 are compared with the Charades action annotations for the same clip (saved by 01b as the oracle input). An event matches an action when their temporal IoU is at least `TIOU` (0.3). *Time only* ignores the words; *time + words* also needs at least one content word in common (e.g. "picks up cup" vs "Taking a cup/glass/bottle from somewhere"). Precision = matched events / events, recall = matched actions / actions, each action matched at most once.

**Detection (VidOR).** Where VidOR box annotations exist (NExT-QA dev videos, saved by 01), detections from notebook 02 are compared with the ground truth once per second: AP at IoU 0.5 for *person* (VidOR adult, child, baby) and class-agnostic for all objects. Boxes are rescaled with both frame sizes. NExT-QA test videos have no public boxes, so this covers dev videos only.

**Tracks.** Counts and lengths from notebook 02 (descriptive). HOTA is not computed: Action Genome has no track identities and VidOR covers only the dev videos.

In [12]:
CVP = sorted(P['cv'].glob('0[23]*_shard*.zip'), key=zip_time, reverse=True)
def cv_file(kind, key, _idx={}):
    """The newest copy of <kind>/<key>.json inside the CV packs, or None."""
    if not _idx:
        for z in CVP:
            try:
                with zipfile.ZipFile(z) as zf:
                    for m in zf.namelist():
                        _idx.setdefault(m, z)
            except (zipfile.BadZipFile, OSError):
                pass
    z = _idx.get(f'{kind}/{key}.json')
    if z is None:
        return None
    with zipfile.ZipFile(z) as zf:
        return json.loads(zf.read(f'{kind}/{key}.json'))

STOP = {'a', 'an', 'the', 'some', 'something', 'somewhere', 'from', 'on', 'in', 'into', 'with', 'of', 'at', 'to', 'up',
        'person', 'someone', 'none', 'and', 'or', 'it', 'is', 'his', 'her'}
def words(s):
    out = set()
    for w in re.findall(r'[a-z]+', str(s).lower().replace('_', ' ')):
        for suf in ('ing', 'es', 's', 'ed'):
            if w.endswith(suf) and len(w) > len(suf) + 2:
                w = w[:-len(suf)]; break
        out.add(w)
    return out - STOP
def tiou(a, b):
    inter = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
    union = max(a[1], b[1]) - min(a[0], b[0])
    return inter / union if union > 0 else 0.0
def match_events(cv, gt, need_words):
    used, tp = set(), 0
    for e in sorted(cv, key=lambda x: x['t0']):
        best, bj = 0.0, None
        for j, g in enumerate(gt):
            if j in used:
                continue
            t = tiou((e['t0'], e['t1']), (g['start'], g['end']))
            if t >= TIOU and t > best and (not need_words or words(e['text']) & words(g['event'])):
                best, bj = t, j
        if bj is not None:
            used.add(bj); tp += 1
    return tp

CVM = {}
orc_dir = P['data'] / 'star' / 'oracle'
tp = {False: 0, True: 0}; n_cv = n_gt = clips = 0
if len(SQ) and orc_dir.exists():
    for uid in sorted(set(SQ.index)):
        orc = load_json(orc_dir / f'{uid}.json', default=None)
        ev = cv_file('events', f'{uid}__orig')
        if orc is None or ev is None:
            continue
        cv = [{'t0': float(e['t0']), 't1': float(e['t1']),
               'text': f"{e.get('verb', '')} {e.get('object_text') or e.get('object') or ''}"} for e in ev['events']
              if (e.get('verb') or '').strip()]
        gt = orc['events']
        clips += 1; n_cv += len(cv); n_gt += len(gt)
        for w in (False, True):
            tp[w] += match_events(cv, gt, w)
    for w, name in [(False, 'time_only'), (True, 'time_and_words')]:
        p = tp[w] / n_cv if n_cv else np.nan; r = tp[w] / n_gt if n_gt else np.nan
        CVM[f'star_events_{name}'] = {'precision': p, 'recall': r, 'f1': 2 * p * r / (p + r) if p and r else 0.0,
                                      'tiou': TIOU, 'clips': clips, 'cv_events': n_cv, 'gt_actions': n_gt}
    print(f'STAR events vs Charades actions on {clips} clips ({n_cv} CV events, {n_gt} ground-truth actions):')
    for k in ['star_events_time_only', 'star_events_time_and_words']:
        if k in CVM:
            v = CVM[k]; print(f"  {k}: precision {v['precision']:.3f}  recall {v['recall']:.3f}  F1 {v['f1']:.3f}")

# detection on VidOR (dev videos)
def ap50(dets, n_gt):
    """VOC-style AP: dets = [(score, is_tp)]."""
    if n_gt == 0 or not dets:
        return np.nan
    dets = sorted(dets, key=lambda x: -x[0])
    tps = np.cumsum([d[1] for d in dets]); fps_ = np.cumsum([not d[1] for d in dets])
    rec = tps / n_gt; prec = tps / np.maximum(1, tps + fps_)
    mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[0], prec, [0]])
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0.0
PERSON_V = {'adult', 'child', 'baby'}
vdir = P['ann'] / 'vidor'
det = {'person': [], 'any': []}; ngt = {'person': 0, 'any': 0}; vids = 0
if vdir.exists() and len(NQ):
    dev_vids = set(read_set('dev').video_id) if len(read_set('dev')) else set()
    for f in sorted(vdir.glob('*.json')):
        vid = f.stem
        if vid not in dev_vids:
            continue
        gtj = load_json(f, default=None); trk = cv_file('tracks', vid); fi = cv_file('frames_index', vid)
        if not gtj or not trk or not fi:
            continue
        cats = {o['tid']: o['category'] for o in gtj.get('subject/objects', [])}
        sx = fi['width'] / gtj.get('width', fi['width']); sy = fi['height'] / gtj.get('height', fi['height'])
        fps = gtj.get('fps', 30.0) or 30.0
        frames = {round(fr['t'], 1): fr for fr in trk['frames']}
        vids += 1
        for t in sorted({round(x) for x in frames}):
            ours = frames.get(round(float(t), 1))
            gi = int(round(t * fps))
            if ours is None or gi >= len(gtj.get('trajectories', [])):
                continue
            gts = [(cats.get(b['tid']), [b['bbox']['xmin'] * sx, b['bbox']['ymin'] * sy, b['bbox']['xmax'] * sx, b['bbox']['ymax'] * sy])
                   for b in gtj['trajectories'][gi]]
            for kind in ('person', 'any'):
                g = [bb for c, bb in gts if kind == 'any' or c in PERSON_V]
                p = [(b.get('conf', 1.0), b['xyxy']) for b in ours['boxes'] if kind == 'any' or b['cls'] == 'person']
                ngt[kind] += len(g); used = set()
                for s, bb in sorted(p, key=lambda x: -x[0]):
                    best, bj = 0.5, None
                    for j, gb in enumerate(g):
                        if j not in used and iou(bb, gb) >= best:
                            best, bj = iou(bb, gb), j
                    if bj is not None:
                        used.add(bj)
                    det[kind].append((s, bj is not None))
    if vids:
        CVM['vidor_detection'] = {'videos': vids, 'person_ap50': ap50(det['person'], ngt['person']),
                                  'class_agnostic_ap50': ap50(det['any'], ngt['any']),
                                  'gt_person_boxes': ngt['person'], 'gt_boxes': ngt['any']}
        v = CVM['vidor_detection']
        print(f"VidOR detection on {vids} dev videos: person AP50 {v['person_ap50']:.3f}, class-agnostic AP50 {v['class_agnostic_ap50']:.3f}")
if 'vidor_detection' not in CVM:
    print('VidOR ground truth not available for dev videos: detection AP skipped')

# track statistics (all processed videos in the evaluated sets)
keys = set(A.key.dropna()) | {f'{u}__orig' for u in SQ.index}
ts = []
for k in sorted(keys):
    trk = cv_file('tracks', k)
    if trk:
        tr = trk['tracks'].values()
        ts.append({'tracks': len(trk['tracks']), 'short_share': np.mean([t.get('short', False) for t in tr]) if tr else 0,
                   'mean_len_s': np.mean([t['t1'] - t['t0'] for t in tr]) if tr else 0,
                   'persons': sum(t['cls'] == 'person' for t in tr)})
if ts:
    TS = pd.DataFrame(ts)
    CVM['tracks'] = {'videos': len(TS), 'tracks_per_video': TS.tracks.mean(), 'persons_per_video': TS.persons.mean(),
                     'short_track_share': TS.short_share.mean(), 'mean_track_len_s': TS.mean_len_s.mean()}
    CVM['tracks'] = {k: float(v) for k, v in CVM['tracks'].items()}
    print('tracks:', {k: round(v, 2) for k, v in CVM['tracks'].items()})
tmp = WORK / 'cv_metrics.json'
save_json(CVM, tmp); _ = shutil.copyfile(tmp, OUT / 'cv_metrics.json')

STAR events vs Charades actions on 360 clips (4290 CV events, 2659 ground-truth actions):
  star_events_time_only: precision 0.440  recall 0.710  F1 0.543
  star_events_time_and_words: precision 0.179  recall 0.288  F1 0.220
VidOR detection on 92 dev videos: person AP50 0.799, class-agnostic AP50 0.603
tracks: {'videos': 539.0, 'tracks_per_video': 29.75, 'persons_per_video': 3.46, 'short_track_share': 0.37, 'mean_track_len_s': 4.17}


## 8. Save the tidy answer table and print the paper's main table

In [13]:
keep = ['run', 'qset', 'format', 'strategy', 'model', 'uid', 'base_uid', 'version', 'dataset', 'video', 'group', 'shard',
        'pred', 'pred_lenient', 'gold', 'correct', 'correct_lenient', 'valid_json', 'calls', 'prompt_tokens', 'gen_tokens',
        'seconds', 'mem_tokens']
for c in ['verifier_supported', 'symbolic_ok', 'reanswered']:
    if c in A:
        keep.append(c)
TA = A.assign(**{k: A.run.map(lambda r, k=k: parse_run(r)[k]) for k in ['qset', 'format', 'strategy']})[keep]
save_csv(TA, 'answers_all.csv')

def row(rid):
    r = QR[QR.run == rid]
    if not len(r):
        return '—'
    r = r.iloc[0]
    return f"{r.accuracy:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}] (n={r.n})"
MAIN = [('Blind LLM (question only)', 'main.BLIND.A', 'star.BLIND.A'),
        ('Captions + A (CAP)', 'main.CAP.A', 'star.CAP.A'),
        ('VideoAgent-style memory + A (VA)', 'main.VA.A', 'star.VA.A'),
        ('M1 raw facts + D', 'main.M1.D', 'star.M1.D'),
        ('M4 event memory + A', 'main.M4.A', 'star.M4.A'),
        ('M4 event memory + D', 'main.M4.D', 'star.M4.D'),
        ('Oracle M4 + D (perfect perception)', '', 'star.ORACLE_M4.D'),
        ('Qwen3-4B + M4 + A', 'main.M4.A.q4b', 'star.M4.A.q4b'),
        ('Qwen3-4B QLoRA + M4 + A', 'main.M4.A.q4b-lora', 'star.M4.A.q4b-lora'),
        ('VLM on 16 frames (no memory)', 'main.VLM.F16', 'star.VLM.F16')]
PT = pd.DataFrame([{'system': s, 'NExT-QA main': row(a) if a else '—', 'STAR': row(b)} for s, a, b in MAIN])
save_csv(PT, 'paper_main_table.csv')
print(PT.to_string(index=False))
RUN_LOG.finish('done', notes=f'{len(A)} answers, {A.run.nunique()} runs')
print('\nfiles written to', OUT, ':', ', '.join(sorted(p.name for p in OUT.glob('*.csv')) + ['cv_metrics.json']))

                            system                 NExT-QA main                         STAR
         Blind LLM (question only) 0.420 [0.343, 0.495] (n=193) 0.305 [0.241, 0.370] (n=190)
                Captions + A (CAP) 0.500 [0.398, 0.594] (n=100)  0.415 [0.315, 0.516] (n=94)
  VideoAgent-style memory + A (VA) 0.500 [0.400, 0.608] (n=100)  0.415 [0.323, 0.521] (n=94)
                  M1 raw facts + D 0.332 [0.266, 0.392] (n=193) 0.474 [0.402, 0.553] (n=190)
               M4 event memory + A 0.544 [0.470, 0.617] (n=193) 0.516 [0.445, 0.589] (n=190)
               M4 event memory + D 0.575 [0.503, 0.643] (n=193) 0.537 [0.464, 0.608] (n=190)
Oracle M4 + D (perfect perception)                            —  0.915 [0.857, 0.968] (n=94)
                 Qwen3-4B + M4 + A 0.456 [0.378, 0.527] (n=193) 0.458 [0.382, 0.534] (n=190)
           Qwen3-4B QLoRA + M4 + A 0.549 [0.477, 0.620] (n=193) 0.537 [0.463, 0.610] (n=190)
      VLM on 16 frames (no memory) 0.829 [0.774, 0.881] (n=193) 0.758 

## What to send back

The tables printed in sections 2, 5, 6 and 7, and the main table in section 8.